# Lectura de paquetes y data

In [45]:
import warnings
import os
import time
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from statsmodels.tsa.seasonal import STL

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet
from sklearn.neighbors import KNeighborsRegressor

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_absolute_error, mean_squared_error

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)  # Reduce verbosity de Optuna


# Agrega todo el directorio padre al path
sys.path.append(os.path.abspath(".."))
from src.utils_ml import ml_training_utils as ml_utils
from src.utils_ml import ml_feature_engineering as fe_utils
from src.utils_ml import ml_plotting as plot_utils

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore")

Utilizamos paths relativos para la lectura de la data

In [ ]:
filename = "ml_pipeline_p_sku.ipynb"  # nombre del archivo actual
print(f"Current absolute path: {os.getcwd()}\n")

# Especificamos la ruta del directorio actual y los directorios de datos y salida
ACTUAL_DIR = os.path.dirname(os.path.abspath(filename))
BASE_DIR = os.path.dirname(ACTUAL_DIR)
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "output")

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

In [ ]:
# Cargar el archivo de Excel
file_path = os.path.join(DATA_DIR, "data_demanda.xlsx")
df_base = pd.read_excel(file_path, sheet_name="data")
df_base = df_base.drop("Cliente", axis=1)

df_base.shape

In [ ]:
df_base.head(5)

In [ ]:
# Filtrar los datos relevantes para este analisis

df = (
    df_base[["Fe.prefer.entrega", "SKU", "Pedidos"]]
    .copy()
    .rename(
        columns={
            "Fe.prefer.entrega": "Fecha",
        }
    )
)
df["Pedidos"] = pd.to_numeric(df["Pedidos"], errors="coerce")

In [ ]:
df

# Preparación de la data

In [ ]:
### Primero, nos aseguramos de que se cuente un dato por SKU por dia
# -------

# rango completo de fechas desde la más antigua hasta la más reciente
fecha_min = df["Fecha"].min()
fecha_max = df["Fecha"].max()
rango_fechas = pd.date_range(start=fecha_min, end=fecha_max, freq="D")

# Obtenemos todos los SKUs únicos
skus = df["SKU"].unique()

# DataFrame con todas las combinaciones de SKU y fecha
combinaciones_completas = pd.MultiIndex.from_product(
    [rango_fechas, skus], names=["Fecha", "SKU"]
).to_frame(index=False)

# Unir con el dataframe original para rellenar con ceros donde falten datos
df_completo = combinaciones_completas.merge(df, on=["Fecha", "SKU"], how="left")

# Rellenar valores faltantes de pedidos con 0
df_completo["Pedidos"] = df_completo["Pedidos"].fillna(0).astype(int)

# Ordenar por SKU y Fecha (opcional)
df_completo = df_completo.sort_values(["SKU", "Fecha"]).reset_index(drop=True)

df = df_completo.copy()

In [ ]:
# Modificar nombre de columnas
df.columns = df.columns.str.replace(".", "_", regex=False).str.lower()

In [ ]:
df.shape

# EDA

## general

In [ ]:
df.isna().sum()

In [ ]:
# porcentaje de ceros por sku
porcentaje_ceros = (
    df.groupby("sku")["pedidos"]
    .apply(lambda x: (x == 0).mean() * 100)
    .reset_index(name="prct_ceros")
    .round(2)
)

# promedio, mediana y desviacion estandar por sku excluyendo ceros
df_temp = df[df["pedidos"] > 0].copy()
promedio = df_temp.groupby("sku")["pedidos"].mean().reset_index(name="Promedio").round()
mediana = df_temp.groupby("sku")["pedidos"].median().reset_index(name="Mediana").round()
desviacion = (
    df_temp.groupby("sku")["pedidos"].std().reset_index(name="Desviacion").round()
)
maximo = df_temp.groupby("sku")["pedidos"].max().reset_index(name="Maximo").round()

# Porcentaje de valores outliers por SKU excluyendo ceros
porcentaje_outliers = (
    df_temp.groupby("sku")["pedidos"]
    .apply(fe_utils.calcular_outliers_porcentaje)
    .reset_index(name="prct_outliers")
    .round(2)
)

# Unir las tablas
tabla_total = pd.merge(porcentaje_ceros, porcentaje_outliers, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, promedio, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, mediana, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, desviacion, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, maximo, on="sku", how="outer")
tabla_total.sort_values(by="prct_ceros", ascending=False)

## Tendencias

In [ ]:
sku = "SKU5"
print(f"Analizando el SKU: {sku}")

In [ ]:
df_sku = df[df["sku"] == sku].copy()
df_sku = df_sku.drop("sku", axis=1)

# graficamos la serie de tiempo del SKU seleccionado usando plotly
fig = px.line(
    df_sku,
    x="fecha",
    y="pedidos",
    title=f"Serie de tiempo de Pedidos para {sku}:",
)
fig.update_layout(
    xaxis_title="Fecha",
)
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="LightGray")
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="LightGray")
fig.update_traces(line=dict(color="blue", width=2))
fig.show()


### Descomposición STL

In [ ]:
# graficamos la tendencia y estacionalidad de cada SKU usando STL
plot_utils.graficar_serie_con_descomposicion(df_sku, sku=sku, periodo=7)


# Feature engineering

## Variables temporales

In [ ]:
df = fe_utils.create_temporal_features(df, "fecha")
df.shape, df.columns

## Variables tipo lag

In [ ]:
df = fe_utils.create_lag_features(
    df, "pedidos", "sku", "fecha", max_daily_lag=14, weekday_lags=3
)
df.shape, df.columns

## Variables tipo promedio moviles

In [ ]:
df = fe_utils.create_rolling_features(df, "pedidos", "sku", "fecha")
df.shape, df.columns

## Variables lags de STL

In [ ]:
df = fe_utils.create_stl_features(
    df, "pedidos", "sku", "fecha", seasonal=7, stl_lags=14
)
df.shape, df.columns


## Aplanamiento de outliers en demanda

In [ ]:
df = fe_utils.cap_upper_outliers(df, "pedidos", "sku")

## Ajustes finales a la data

In [ ]:
# Eliminamos todas las filas con valores NaN para que no afecten el entrenamiento
df.dropna(inplace=True)
df.shape, df.columns

In [ ]:
df

# Modelling 

In [ ]:
## Preparación de data para modelling

# Separamos el conjunto de datos en entrenamiento y prueba, usando los últimos 7 días como prueba
test = df.tail(7)
df_mod = df[:-7].copy()

# Identificamos las variables predictoras
features = df_mod.columns.difference(["fecha", "sku", "pedidos"]).to_list()

# definimos el numero de ventanas de evaluación y el tamaño de las ventanas
val_iter = 3
val_size = 7

## Evaluación modelos XGBoost

In [ ]:
# Creamos un DataFrame para almacenar los resultados de cada SKU
# Este DataFrame contendrá el SKU, los parámetros del modelo y las métricas de evaluación
##########

results = []

for sku in df_mod["sku"].unique():
    print("\n-----------------------")
    print(f"\n🔍 Optimizando para SKU: {sku}\n")

    # Filtramos el DataFrame para el SKU actual
    df_sku = df_mod[df_mod["sku"] == sku].copy()

    # Calculamos el tamaño del conjunto de entrenamiento
    train_size = df_sku.shape[0] - val_iter * val_size

    # Definimos el espacio de búsqueda de hiperparámetros para el modelo XGBoost
    # Usamos funciones lambda para que Optuna pueda sugerir valores
    param_grid = {
        "max_depth": lambda trial: trial.suggest_int("max_depth", 1, 8, step=1),
        "learning_rate": lambda trial: trial.suggest_float(
            "learning_rate", 0.001, 0.1, step=0.001
        ),
        "n_estimators": lambda trial: trial.suggest_int(
            "n_estimators", 50, 1500, step=50
        ),
        "subsample": lambda trial: trial.suggest_float(
            "subsample", 0.5, 1.0, step=0.02
        ),
        "colsample_bytree": lambda trial: trial.suggest_float(
            "colsample_bytree", 0.5, 1.0, step=0.02
        ),
        "gamma": lambda trial: trial.suggest_float("gamma", 1, 30, step=0.5),
        "reg_alpha": lambda trial: trial.suggest_float("reg_alpha", 1, 30, step=0.5),
        "reg_lambda": lambda trial: trial.suggest_float("reg_lambda", 1, 30, step=0.5),
        "min_child_weight": lambda trial: trial.suggest_int(
            "min_child_weight", 5, 20, step=1
        ),
        "random_state": 100,  # Fijamos la semilla para reproducibilidad
    }

    # Optimizamos el modelo XGBoost usando Optuna con ventana recursiva
    study = ml_utils.optimize_model_with_optuna(
        model_class=XGBRegressor,
        param_grid=param_grid,
        X=df_sku[features],
        y=df_sku["pedidos"],
        n_trials=125,
        val_iter=val_iter,
        train_size=train_size,
        val_size=val_size,
        show_progress_bar=True,
    )

    # Obtenemos el mejor trial del estudio
    # y almacenamos los resultados en un diccionario
    best_trial = study.best_trials[0]
    results.append(
        {
            "sku": sku,
            "study": study,
            "best_params": best_trial.params,
            "best_smape": best_trial.values[0],
            "best_gap": best_trial.values[1],
            "model": "XGBRegressor",
            "n_trials": len(study.trials),
        }
    )

    print("\n-----------------------")


df_results_xgboost = pd.DataFrame(results)

In [ ]:
df_results_xgboost

In [ ]:
## Mostrar graficas de Optuna por estudio especifico

# sku = "SKU1"
# temp = df_results_xgboost[df_results_xgboost["sku"] == sku].copy()
# plot_utils.mostrar_graficas_optuna(temp["study"], temp["model"])

## Evaluación modelos Random Forest

In [ ]:
# Creamos un DataFrame para almacenar los resultados de cada SKU
# Este DataFrame contendrá el SKU, los parámetros del modelo y las métricas de evaluación
##########

results = []

for sku in df_mod["sku"].unique():
    print("\n-----------------------")
    print(f"🔍 Optimizando para SKU: {sku}\n")

    # Filtramos por SKU
    df_sku = df_mod[df_mod["sku"] == sku].copy()

    # Tamaño de entrenamiento
    train_size = df_sku.shape[0] - val_iter * val_size

    # Espacio de búsqueda de hiperparámetros para Random Forest
    param_grid = {
        "n_estimators": lambda trial: trial.suggest_int(
            "n_estimators", 50, 1000, step=50
        ),
        "max_depth": lambda trial: trial.suggest_int("max_depth", 1, 10, step=1),
        "min_samples_split": lambda trial: trial.suggest_int(
            "min_samples_split", 2, 10
        ),
        "min_samples_leaf": lambda trial: trial.suggest_int(
            "min_samples_leaf", 3, 15, step=1
        ),
        "max_features": lambda trial: trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None]
        ),
        "bootstrap": lambda trial: trial.suggest_categorical(
            "bootstrap", [True, False]
        ),
        "random_state": lambda trial: 100,  # Fijamos el random_state para reproducibilidad
    }

    # Optimizamos usando tu función personalizada con modelo RandomForestRegressor
    study = ml_utils.optimize_model_with_optuna(
        model_class=RandomForestRegressor,
        param_grid=param_grid,
        X=df_sku[features],
        y=df_sku["pedidos"],
        n_trials=125,
        val_iter=val_iter,
        train_size=train_size,
        val_size=val_size,
        show_progress_bar=True,
    )

    # Registramos los resultados
    best_trial = study.best_trials[0]
    results.append(
        {
            "sku": sku,
            "study": study,
            "best_params": best_trial.params,
            "best_smape": best_trial.values[0],
            "best_gap": best_trial.values[1],
            "model": "RandomForestRegressor",
            "n_trials": len(study.trials),
        }
    )

    print("-----------------------")

# Creamos DataFrame con resultados
df_results_rf = pd.DataFrame(results)


In [ ]:
df_results_rf

In [ ]:
## Mostrar graficas de Optuna por estudio especifico

# sku = "SKU1"
# temp = df_results_rf[df_results_rf["sku"] == sku].copy()
# plot_utils.mostrar_graficas_optuna(temp["study"], temp["model"])

## Evaluación modelos Elastic Net

In [ ]:
# Creamos un DataFrame para almacenar los resultados de cada SKU
# Este DataFrame contendrá el SKU, los parámetros del modelo y las métricas de evaluación
##########

results = []

for sku in df_mod["sku"].unique():
    print("\n-----------------------")
    print(f"🔍 Optimizando ElasticNet para SKU: {sku}\n")

    # Filtramos por SKU
    df_sku = df_mod[df_mod["sku"] == sku].copy()
    train_size = df_sku.shape[0] - val_iter * val_size

    # Espacio de búsqueda de hiperparámetros para ElasticNet
    param_grid = {
        "model__alpha": lambda trial: trial.suggest_float(
            "model__alpha", 0.0001, 1000.0, log=True
        ),
        "model__l1_ratio": lambda trial: trial.suggest_float(
            "model__l1_ratio", 0.0, 1.0, step=0.01
        ),  # 0 = Ridge, 1 = Lasso
        "model__random_state": lambda trial: 100,  # Fijamos el random_state para reproducibilidad
    }

    # Pipeline: escalado + modelo
    model_pipeline = Pipeline(
        [("scaler", StandardScaler()), ("model", ElasticNet(max_iter=20000))]
    )

    # Optimización con tu función
    study = ml_utils.optimize_model_with_optuna(
        model_class=lambda **params: model_pipeline.set_params(**params),
        param_grid=param_grid,
        X=df_sku[features],
        y=df_sku["pedidos"],
        n_trials=200,
        val_iter=val_iter,
        train_size=train_size,
        val_size=val_size,
        show_progress_bar=True,
    )

    best_trial = study.best_trials[0]
    results.append(
        {
            "sku": sku,
            "study": study,
            "best_params": best_trial.params,
            "best_smape": best_trial.values[0],
            "best_gap": best_trial.values[1],
            "model": "ElasticNet",
            "n_trials": len(study.trials),
        }
    )

    print("-----------------------")

df_results_elastic = pd.DataFrame(results)


In [ ]:
df_results_elastic

In [ ]:
## Mostrar graficas de Optuna por estudio especifico

# sku = "SKU1"
# temp = df_results_elastic[df_results_elastic["sku"] == sku].copy()
# plot_utils.mostrar_graficas_optuna(temp["study"], temp["model"])

## Evaluación modelos KNeighbors 

In [53]:
# Creamos un DataFrame para almacenar los resultados de cada SKU
# Este DataFrame contendrá el SKU, los parámetros del modelo y las métricas de evaluación
##########

results = []

for sku in df_mod["sku"].unique()[0:2]:
    print("\n-----------------------")
    print(f"🔍 Optimizando KNN para SKU: {sku}\n")

    df_sku = df_mod[df_mod["sku"] == sku].copy()
    train_size = df_sku.shape[0] - val_iter * val_size

    # Pipeline: escalado obligatorio + modelo
    pipeline = Pipeline(
        [("scaler", StandardScaler()), ("model", KNeighborsRegressor())]
    )

    # Espacio de búsqueda
    param_grid = {
        "model__n_neighbors": lambda trial: trial.suggest_int(
            "model__n_neighbors", 2, 30, step=1
        ),
        "model__weights": lambda trial: trial.suggest_categorical(
            "model__weights", ["uniform", "distance"]
        ),
        "model__p": lambda trial: trial.suggest_int(
            "model__p", 1, 2
        ),  # 1 = manhattan, 2 = euclidean
        "model__leaf_size": lambda trial: trial.suggest_int(
            "model__leaf_size", 5, 100, step=1
        ),
        "model__algorithm": lambda trial: trial.suggest_categorical(
            "model__algorithm", ["auto", "ball_tree", "kd_tree"]
        ),
    }

    # Llamada a tu función personalizada
    study = ml_utils.optimize_model_with_optuna(
        model_class=lambda **params: pipeline.set_params(**params),
        param_grid=param_grid,
        X=df_sku[features],
        y=df_sku["pedidos"],
        n_trials=300,
        val_iter=val_iter,
        train_size=train_size,
        val_size=val_size,
        show_progress_bar=True,
    )

    best_trial = study.best_trials[0]
    results.append(
        {
            "sku": sku,
            "study": study,
            "best_params": best_trial.params,
            "best_smape": best_trial.values[0],
            "best_gap": best_trial.values[1],
            "model": "KNeighborsRegressor",
            "n_trials": len(study.trials),
        }
    )

    print("-----------------------")

df_results_knn = pd.DataFrame(results)



-----------------------
🔍 Optimizando KNN para SKU: SKU1



100%|██████████| 300/300 [00:05<00:00, 55.26it/s]



📌 Mejores Trials:
Trial 0 - SMAPE: 44.0000, GAP: 5.9000
Trial 1 - SMAPE: 36.0000, GAP: 13.8800
Trial 2 - SMAPE: 36.0000, GAP: 13.8800
Trial 3 - SMAPE: 36.0000, GAP: 13.8800
Trial 4 - SMAPE: 30.7700, GAP: 24.4100
Trial 5 - SMAPE: 30.7700, GAP: 24.4100
Trial 6 - SMAPE: 28.4400, GAP: 32.1300
Trial 7 - SMAPE: 36.0000, GAP: 13.8800
Trial 8 - SMAPE: 36.0000, GAP: 13.8800
Trial 9 - SMAPE: 44.0000, GAP: 5.9000
Trial 10 - SMAPE: 30.7700, GAP: 24.4100
Trial 11 - SMAPE: 28.4400, GAP: 32.1300
Trial 12 - SMAPE: 36.0000, GAP: 13.8800
Trial 13 - SMAPE: 30.8300, GAP: 22.0700
Trial 14 - SMAPE: 30.8300, GAP: 22.0700
Trial 15 - SMAPE: 30.8300, GAP: 22.0700
Trial 16 - SMAPE: 44.0000, GAP: 5.9000
Trial 17 - SMAPE: 44.0000, GAP: 5.9000
Trial 18 - SMAPE: 44.0000, GAP: 5.9000
Trial 19 - SMAPE: 28.6400, GAP: 28.6400
Trial 20 - SMAPE: 28.6400, GAP: 28.6400
Trial 21 - SMAPE: 30.8300, GAP: 22.0700
Trial 22 - SMAPE: 28.6400, GAP: 28.6400
Trial 23 - SMAPE: 30.8300, GAP: 22.0700
Trial 24 - SMAPE: 28.6400, GAP: 28.6

100%|██████████| 300/300 [00:05<00:00, 54.58it/s]


📌 Mejores Trials:
Trial 0 - SMAPE: 33.8300, GAP: 5.3200
Trial 1 - SMAPE: 33.8300, GAP: 5.3200
Trial 2 - SMAPE: 33.8300, GAP: 5.3200
Trial 3 - SMAPE: 33.8300, GAP: 5.3200
Trial 4 - SMAPE: 32.5700, GAP: 5.4200
Trial 5 - SMAPE: 32.5700, GAP: 5.4200
Trial 6 - SMAPE: 31.6100, GAP: 6.8500
Trial 7 - SMAPE: 31.6100, GAP: 6.8500
Trial 8 - SMAPE: 30.2300, GAP: 7.9400
Trial 9 - SMAPE: 31.6100, GAP: 6.8500
Trial 10 - SMAPE: 31.6100, GAP: 6.8500
Trial 11 - SMAPE: 31.6100, GAP: 6.8500
Trial 12 - SMAPE: 31.6100, GAP: 6.8500
Trial 13 - SMAPE: 30.2300, GAP: 7.9400
Trial 14 - SMAPE: 31.6100, GAP: 6.8500
Trial 15 - SMAPE: 32.5700, GAP: 5.4200
Trial 16 - SMAPE: 30.2300, GAP: 7.9400
Trial 17 - SMAPE: 31.6100, GAP: 6.8500
Trial 18 - SMAPE: 31.3200, GAP: 7.1000
Trial 19 - SMAPE: 31.3200, GAP: 7.1000
Trial 20 - SMAPE: 30.2300, GAP: 7.9400
Trial 21 - SMAPE: 32.5700, GAP: 5.4200
Trial 22 - SMAPE: 30.2300, GAP: 7.9400
Trial 23 - SMAPE: 31.3200, GAP: 7.1000
Trial 24 - SMAPE: 32.5700, GAP: 5.4200
Trial 25 - SMAPE

In [54]:
df_results_knn

,sku,study,best_params,best_smape,best_gap,model,n_trials
0,SKU1,<optuna.study.study.Study object at 0x13824e000>,"{'model__n_neighbors': 2, 'model__weights': 'u...",44.00,5.90,KNeighborsRegressor,300
1,SKU10,<optuna.study.study.Study object at 0x12cf50290>,"{'model__n_neighbors': 8, 'model__weights': 'u...",33.83,5.32,KNeighborsRegressor,300


In [ ]:
## Mostrar graficas de Optuna por estudio especifico

# sku = "SKU1"
# temp = df_results_knn[df_results_knn["sku"] == sku].copy()
# plot_utils.mostrar_graficas_optuna(temp["study"], temp["model"])

## Selección mejor modelo y ajuste final

In [48]:
# Eliminamos los estudios de Optuna de todos los DataFrames de resultados
# df_results_xgboost = df_results_xgboost.drop(columns=["study"])
# df_results_rf = df_results_rf.drop(columns=["study"])
# df_results_elastic = df_results_elastic.drop(columns=["study"])
# df_results_knn = df_results_knn.drop(columns=["study"])

# Union de resultados de los modelos
df_results = pd.concat(
    [df_results_xgboost, df_results_rf, df_results_elastic, df_results_knn],
    ignore_index=True,
).sort_values(by=["sku", "best_smape"], ascending=[True, True])

# Por cada SKU, obtenemos el mejor modelo
df_best_models = df_results.loc[
    df_results.groupby("sku")["best_smape"].idxmin()
].reset_index(drop=True)


# Ordenamos por mejor SMAPE
df_best_models = df_best_models.sort_values(by="best_smape", ascending=True)

In [49]:
df_results

,sku,study,best_params,best_smape,best_gap,model,n_trials
4,SKU1,<optuna.study.study.Study object at 0x12ce2e250>,"{'model__alpha': 0.041858227295469716, 'model_...",27.69,23.54,ElasticNet,200
0,SKU1,<optuna.study.study.Study object at 0x12b294980>,"{'max_depth': 6, 'learning_rate': 0.032, 'n_es...",29.86,2.52,XGBRegressor,120
2,SKU1,<optuna.study.study.Study object at 0x12b24cb90>,"{'n_estimators': 950, 'max_depth': 6, 'min_sam...",36.52,6.37,RandomForestRegressor,100
6,SKU1,<optuna.study.study.Study object at 0x12de86300>,"{'model__n_neighbors': 4, 'model__weights': 'u...",38.00,20.54,KNeighborsRegressor,15
3,SKU10,<optuna.study.study.Study object at 0x12d30f6f0>,"{'n_estimators': 300, 'max_depth': 9, 'min_sam...",28.02,3.15,RandomForestRegressor,100
1,SKU10,<optuna.study.study.Study object at 0x12b2274d0>,"{'max_depth': 1, 'learning_rate': 0.0520000000...",29.15,1.27,XGBRegressor,120
7,SKU10,<optuna.study.study.Study object at 0x12ce799a0>,"{'model__n_neighbors': 10, 'model__weights': '...",32.77,6.80,KNeighborsRegressor,15
5,SKU10,<optuna.study.study.Study object at 0x12cdd8050>,"{'model__alpha': 148.9838559398634, 'model__l1...",42.30,4.59,ElasticNet,200


In [50]:
df_best_models

,sku,study,best_params,best_smape,best_gap,model,n_trials
0,SKU1,<optuna.study.study.Study object at 0x12ce2e250>,"{'model__alpha': 0.041858227295469716, 'model_...",27.69,23.54,ElasticNet,200
1,SKU10,<optuna.study.study.Study object at 0x12d30f6f0>,"{'n_estimators': 300, 'max_depth': 9, 'min_sam...",28.02,3.15,RandomForestRegressor,100


# Predicción y graficas 

In [ ]:
# # Reentrenamos la serie de tiempo de cada SKU con el mejor modelo, usando el conjunto de entrenamiento completo
# for sku in df_best_models["sku"].unique():
#     print(f"\n🔍 Reentrenando modelo para SKU: {sku}\n")

#     # Filtramos el DataFrame para el SKU actual
#     df_sku = df[df["sku"] == sku].copy()

#     # Definimos las variables predictoras
#     X = df_sku[features]
#     y = df_sku["pedidos"]

#     # Obtenemos el mejor modelo y sus parámetros
#     best_model_name = df_best_models.loc[df_best_models["sku"] == sku, "model"].values[
#         0
#     ]
#     best_params = df_best_models.loc[
#         df_best_models["sku"] == sku, "best_params"
#     ].values[0]

#     # Creamos el modelo con los mejores parámetros
#     if best_model_name == "XGBRegressor":
#         model = XGBRegressor(**best_params)
#     elif best_model_name == "RandomForestRegressor":
#         model = RandomForestRegressor(**best_params)
#     elif best_model_name == "ElasticNet":
#         model = Pipeline(
#             [
#                 ("scaler", StandardScaler()),
#                 ("model", ElasticNet(**best_params)),
#             ]
#         )
#     else:
#         raise ValueError(f"Modelo desconocido: {best_model_name}")

#     # Entrenamos el modelo con todo el conjunto de entrenamiento
#     model.fit(X, y)

#     # Guardamos el modelo entrenado en un archivo
#     ml_utils.save_model(model, sku, OUTPUT_DIR)
